
# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Build an Interactive Retail Sales Dashboard

### Scenario

You have just joined the analytics team of **UrbanCart**, a mid-sized retail chain with stores across
three regions. Regional managers currently receive a static monthly PDF report and complain that:

1. They can't drill into *why* a number moved without emailing the analytics team.
2. Comparing stores or products side by side means flipping between many separate PDF pages.
3. Spotting a sudden dip in a specific product/store is slow and easy to miss.

**Your job:** apply the interaction techniques from the lecture — dynamic queries, coordinated views
& brushing, table lens, focus+context drill-down, and single-screen dashboard design — to build an
**interactive exploration tool** that solves these three complaints.

### How this notebook is organized

- A sample dataset is generated for you (Task 0) — don't skip running it.
- Each task states the **real-world usage**: which complaint above it solves, and why the technique
  fits.
- Tasks give you a **scaffold** (imports, helper stubs, expected output) — **you write the core logic**
  marked with `# TODO`. Do not just copy the Demo notebook's code verbatim — the data shape, column
  names and the exact question being asked are different here, so you will need to adapt the pattern,
  not paste it.


> **Grading-relevant tip:** every `# TODO` cell tells you what the final variable/plot must be named
> or show — read it before coding, since later cells depend on it.



## Task 0 — Load the Sample Dataset (provided)

Run the cells below as-is. This generates `sales_df`: 24 months of daily-aggregated sales across
6 stores (3 regions), 10 products (4 categories), including revenue, profit and customer ratings —
enough structure to support every technique you'll build below.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import interact, HBox, VBox

sns.set_theme(style="whitegrid")
np.random.seed(7)
print("Libraries loaded.")


In [ ]:

# --- Sample dataset generator: UrbanCart retail sales (provided, do not need to modify) ---
stores = pd.DataFrame({
    "store_id":   ["S01", "S02", "S03", "S04", "S05", "S06"],
    "store_city": ["Bengaluru", "Mysuru", "Mumbai", "Pune", "Delhi", "Jaipur"],
    "region":     ["South", "South", "West", "West", "North", "North"],
})

products = pd.DataFrame({
    "product":  ["Wireless Earbuds", "Smartwatch", "Laptop Sleeve", "USB-C Hub",
                 "Yoga Mat", "Dumbbell Set", "Running Shoes", "Track Jacket",
                 "Air Fryer", "Blender"],
    "category": ["Electronics", "Electronics", "Electronics", "Electronics",
                 "Fitness", "Fitness", "Apparel", "Apparel",
                 "Home", "Home"],
    "unit_price": [1499, 4999, 899, 1299, 799, 2499, 3499, 1999, 3999, 2299],
})

months = pd.date_range("2023-01-01", periods=24, freq="MS")

rows = []
for _, s in stores.iterrows():
    store_scale = np.random.uniform(0.7, 1.4)          # some stores just sell more
    for _, p in products.iterrows():
        product_scale = np.random.uniform(0.6, 1.6)
        trend = np.random.normal(0.01, 0.01)             # slow month-over-month growth/decline
        for i, m in enumerate(months):
            seasonal = 1 + 0.25 * np.sin(2 * np.pi * (m.month / 12))   # yearly seasonality
            units = max(0, np.random.poisson(15 * store_scale * product_scale * seasonal * (1 + trend) ** i))
            revenue = units * p["unit_price"]
            profit_margin = np.random.uniform(0.12, 0.35)
            profit = revenue * profit_margin
            rating = np.clip(np.random.normal(4.1, 0.4), 1, 5)
            # NOTE: use bracket indexing (p["product"]), not p.product -- pandas Series already has
            # a built-in .product() method, so attribute access on a column literally named "product"
            # silently returns that method instead of your data. This is a common real-world gotcha.
            rows.append((m, s["store_id"], s["store_city"], s["region"], p["product"], p["category"],
                         p["unit_price"], units, revenue, profit, round(rating, 1)))

sales_df = pd.DataFrame(rows, columns=[
    "month", "store_id", "store_city", "region", "product", "category",
    "unit_price", "units_sold", "revenue", "profit", "avg_rating"
])
for _col in ["units_sold", "revenue", "profit"]:
    sales_df[_col] = sales_df[_col].astype(float)  # float, so the anomaly injection below can scale them

# Inject one deliberate "anomaly" for Task 4 (a real dip a manager would want to investigate)
mask = (sales_df.store_id == "S03") & (sales_df["product"] == "Smartwatch") & (sales_df.month == "2024-06-01")
sales_df.loc[mask, ["units_sold", "revenue", "profit"]] = sales_df.loc[mask, ["units_sold", "revenue", "profit"]] * 0.15

print(sales_df.shape)
sales_df.head()



**Reflection (answer in a sentence or two, in this markdown cell):**
Before writing any code, look at the columns above. Which columns would you use as *filters* for
dynamic querying, and which would you use to *link* views together for brushing? Write your answer
here.

_Your answer:_

Filters: `region`, `store_id`, `product`, `category`, `unit_price`, `units_sold`, and `month` are useful dynamic-query fields. For linked views, `store_id`, `product`, `category`, and `month` are the main keys because they let a selection in one view identify the same records in the other views.



## Task 1 — Static vs. Interactive: Which Report Actually Answers the Manager's Question?

**Real-world usage:** Complaint #1 ("flipping through PDF pages"). A regional manager asks:
*"Show me total revenue by product, and let me check exactly how much any bar is worth without
squinting at a printed axis."*

**Why this technique:** A static bar chart is fine for the *headline* number, but it can't answer a
precise follow-up question live — that needs a hover-enabled interactive chart (Slide 4-8).

### Your task
1. Build a **static** `matplotlib`/`seaborn` bar chart of total revenue per `product`, sorted
   descending.
2. Build the **same** chart as an **interactive** `plotly` bar chart where hovering shows the exact
   revenue figure and the category of that product.
3. In the markdown cell below, state in 1-2 sentences: for *this specific* manager request, which
   version wins, and why (tie your answer to target-audience / data-story / ROI from the slide).


In [ ]:
# --- 1a. STATIC chart ---
# TODO:
#   - Group sales_df by "product", sum "revenue"
#   - Sort descending
#   - Plot a horizontal or vertical bar chart with matplotlib/seaborn
#   - Title it clearly

revenue_by_product = (
    sales_df.groupby(["product", "category"], as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)

plt.figure(figsize=(9, 5))
sns.barplot(data=revenue_by_product, x="revenue", y="product")
plt.xlabel("Total Revenue ($)")
plt.ylabel("Product")
plt.title("Total Revenue by Product")
plt.tight_layout()
plt.show()


In [ ]:
# --- 1b. INTERACTIVE chart ---
# TODO:
#   - Use plotly.express.bar on the SAME grouped data
#   - Add a hover field for "category" (hint: you'll need category alongside revenue -- think about
#     what columns your groupby needs to keep)
#   - Title it clearly

fig = px.bar(
    revenue_by_product,
    x="revenue",
    y="product",
    orientation="h",
    hover_data={"category": True, "revenue": ":,.2f"},
    title="Interactive Total Revenue by Product",
    labels={"revenue": "Total Revenue ($)", "product": "Product", "category": "Category"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()



**Your 1-2 sentence answer:**

For this specific request, the interactive Plotly version is more useful because the manager can hover over a product to get its exact revenue and category immediately, without estimating from an axis. The static chart is still useful for the headline ranking, but the interactive version has better ROI for the follow-up question.



## Task 2 — Dynamic Queries: Let Managers Filter Without Asking You

**Real-world usage:** Regional managers want to self-serve answer questions like *"which
products, in what price range, sold above X units last month?"* without emailing the analytics team
every time (Complaint #1 again, but for filtering rather than just reading).

**Why this technique:** Dynamic queries (double-ended range sliders) let the pattern emerge from the
noise live, with the response updating in real time (Slide 23).

### Your task
Build **two** linked `ipywidgets` range sliders:
- `price_slider` — filters on `unit_price`
- `units_slider` — filters on `units_sold`

Write a function `dynamic_filter(price_range, units_range)` that:
1. Filters `sales_df` to rows where `unit_price` is inside `price_range` **and** `units_sold` is
   inside `units_range`.
2. Prints how many rows match out of the total.
3. Shows a scatter plot of `unit_price` vs `units_sold`, with matching rows highlighted in a
   distinct color against all other rows in gray (same pattern as the Demo notebook's dynamic query
   cell — but you must build the filter condition and the plot yourself for these two columns).

Wire it up with `interact()`.


In [ ]:
# TODO: create price_slider (FloatRangeSlider) spanning sales_df.unit_price min..max
price_slider = widgets.FloatRangeSlider(
    value=(float(sales_df["unit_price"].min()), float(sales_df["unit_price"].max())),
    min=float(sales_df["unit_price"].min()),
    max=float(sales_df["unit_price"].max()),
    step=1,
    description="Price:",
    continuous_update=False,
)

# TODO: create units_slider (FloatRangeSlider or IntRangeSlider) spanning sales_df.units_sold min..max
units_slider = widgets.IntRangeSlider(
    value=(int(sales_df["units_sold"].min()), int(sales_df["units_sold"].max())),
    min=int(sales_df["units_sold"].min()),
    max=int(sales_df["units_sold"].max()),
    step=1,
    description="Units:",
    continuous_update=False,
)

def dynamic_filter(price_range, units_range):
    # TODO: filter sales_df using both ranges
    filtered = sales_df[
        sales_df["unit_price"].between(price_range[0], price_range[1])
        & sales_df["units_sold"].between(units_range[0], units_range[1])
    ]

    # TODO: print "X / Y rows match"
    print(f"{len(filtered)} / {len(sales_df)} rows match")

    # TODO: scatter plot: all rows in gray, filtered rows highlighted in color
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(sales_df["unit_price"], sales_df["units_sold"],
               color="lightgray", alpha=0.45, s=18, label="All rows")
    ax.scatter(filtered["unit_price"], filtered["units_sold"],
               color="#4C72B0", alpha=0.75, s=25, label="Matching rows")
    ax.set_xlabel("Unit Price")
    ax.set_ylabel("Units Sold")
    ax.set_title("Dynamic Query: Price and Units Sold")
    ax.legend()
    plt.tight_layout()
    plt.show()

# TODO: call interact(dynamic_filter, price_range=price_slider, units_range=units_slider)
interact(dynamic_filter, price_range=price_slider, units_range=units_slider)



**Reflection:** The slide notes dynamic queries work well "up to ~100,000 points with five or fewer
sliders." `sales_df` has far fewer rows. If UrbanCart grew to 500 stores and this dataset became 50x
larger, would two sliders still be enough, or would you need a different technique from the lecture?
Name one.

_Your answer:_

Two sliders may still help, but at 50× the data size I would move toward a more scalable interactive aggregation or overview-plus-filtering approach, such as a coordinated dashboard with pre-aggregated views. That reduces the amount of raw data the user has to inspect at once.



## Task 3 — Coordinated Views & Cross-View Brushing: Compare Stores Without Flipping Pages

**Real-world usage:** Complaint #2 — "comparing stores means flipping between many PDF pages."
A manager wants to click one store in an overview chart and instantly see that store's category
breakdown and its monthly trend, without losing sight of how it compares to the others.

**Why this technique:** Coordinated multiple views + brushing (Slide 24) — selecting a subset in one
view highlights the same records everywhere else, replacing the working-memory cost of flipping pages.

### Your task
Build **three linked views** for a store the user selects from a dropdown:
1. **Overview bar chart** — total revenue per store (all 6 stores), with the selected store's bar in
   a distinct color and every other store's bar in gray (this is the "muted unselected categories"
   trick from the slide).
2. **Category breakdown** — a pie or bar chart of revenue by `category`, filtered to the selected
   store only.
3. **Monthly trend line** — revenue by `month`, filtered to the selected store only.

Wrap all three in one function driven by a single `ipywidgets.Dropdown` of `store_id` values, and lay
the three charts out together (e.g. with `plt.subplots` for 1+2+3, or separate `display()` calls).


In [ ]:
store_overview = sales_df.groupby("store_id").revenue.sum().reset_index()

def store_coordinated_view(selected_store):
    # TODO 1: bar chart of store_overview, selected store highlighted, others muted gray
    selected_colors = [
        "#4C72B0" if store == selected_store else "lightgray"
        for store in store_overview["store_id"]
    ]

    # TODO 2: category breakdown pie/bar for selected_store only
    selected_df = sales_df[sales_df["store_id"] == selected_store]
    category_overview = selected_df.groupby("category")["revenue"].sum().sort_values(ascending=False)

    # TODO 3: monthly revenue trend for selected_store only
    monthly_overview = (
        selected_df.groupby("month", as_index=False)["revenue"]
        .sum()
        .sort_values("month")
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].bar(store_overview["store_id"], store_overview["revenue"], color=selected_colors)
    axes[0].set_title("Revenue by Store")
    axes[0].set_xlabel("Store")
    axes[0].set_ylabel("Revenue ($)")

    axes[1].bar(category_overview.index, category_overview.values, color="#55A868")
    axes[1].set_title(f"Category Revenue — {selected_store}")
    axes[1].set_xlabel("Category")
    axes[1].set_ylabel("Revenue ($)")
    axes[1].tick_params(axis="x", rotation=30)

    axes[2].plot(monthly_overview["month"], monthly_overview["revenue"], "o-", color="#C44E52")
    axes[2].set_title(f"Monthly Revenue — {selected_store}")
    axes[2].set_xlabel("Month")
    axes[2].set_ylabel("Revenue ($)")
    axes[2].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

# TODO: wire this up with interact() using a Dropdown of stores["store_id"] values
interact(
    store_coordinated_view,
    selected_store=widgets.Dropdown(
        options=stores["store_id"].tolist(),
        value=stores["store_id"].iloc[0],
        description="Store:",
    ),
)



**Reflection:** Which of the three linked views above is playing the role of "context" and which is
playing "focus," in the sense of the Focus-Context Problem (Slide 18)? Is that the same distinction as
brushing, or a different but related idea? Explain briefly.

_Your answer:_

The store overview is the main context view because it keeps all stores visible while the selected store becomes the focus in the category and monthly views. This is related to brushing but not identical: brushing is the selection/highlighting mechanism, while focus+context is the broader design goal of keeping the selected detail understandable against its surrounding context.



## Task 4 — Focus + Context Drill-Down: Investigate the Dip

**Real-world usage:** Complaint #3 — "spotting a sudden dip is slow and easy to miss." One product at
one store has a real anomaly hidden in `sales_df` (planted in Task 0). A manager needs to find it and
drill in **without losing the overview**, using the *cheapest* interaction tier that still answers the
question (Drill-Down Cost Hierarchy, Slide 17).

### Your task
**Part A — find the anomaly.**
Write code that computes, for every `(store_id, product)` pair, the ratio of the *minimum* month's
`units_sold` to the *median* month's `units_sold` across the 24 months. Sort ascending and show the
top 5 most extreme drops. (Hint: `groupby(["store_id","product"]).units_sold.agg(...)` with a custom
function, or compute `min` and `median` separately and combine.)

**Part B — build the drill-down.**
Using **hover** (cheap tier) on an overview chart of all `(store_id, product)` trend lines, let the
user identify the anomalous line. Then, given the `store_id`/`product` you found in Part A, plot its
monthly trend **alongside** a faint gray context line showing the *average* trend across all
stores/products for the same product — so the dip is visible with its context still present (not a
full window replacement).


In [ ]:
# --- Part A: find the anomaly ---
# TODO: compute min/median ratio per (store_id, product) and show the 5 most extreme

pair_stats = (
    sales_df.groupby(["store_id", "product"])["units_sold"]
    .agg(min_units="min", median_units="median")
    .reset_index()
)
pair_stats["min_median_ratio"] = pair_stats["min_units"] / pair_stats["median_units"]
anomaly_ranking = pair_stats.sort_values("min_median_ratio", ascending=True).head(5)

print(anomaly_ranking.to_string(index=False))


In [ ]:
# --- Part B: drill-down with context preserved ---
# TODO 1: build an overview plotly line chart of units_sold over month, one line per (store_id, product)
#         combination, with hover showing store_id + product (cheap-tier epistemic action)

overview = sales_df.copy()
overview["store_product"] = overview["store_id"] + " | " + overview["product"]

overview_fig = px.line(
    overview,
    x="month",
    y="units_sold",
    color="store_product",
    line_group="store_product",
    hover_data=["store_id", "product", "units_sold"],
    title="Units Sold by Store and Product",
    labels={"month": "Month", "units_sold": "Units Sold", "store_product": "Store | Product"},
)
overview_fig.show()

# TODO 2: using the (store_id, product) pair you identified as the top anomaly in Part A,
#         plot its monthly units_sold trend (focus) together with the average trend for that SAME
#         product across all other stores (context), on one shared chart

top_store = anomaly_ranking.iloc[0]["store_id"]
top_product = anomaly_ranking.iloc[0]["product"]

focus = sales_df[
    (sales_df["store_id"] == top_store) & (sales_df["product"] == top_product)
].sort_values("month")

context = (
    sales_df[
        (sales_df["product"] == top_product) & (sales_df["store_id"] != top_store)
    ]
    .groupby("month", as_index=False)["units_sold"]
    .mean()
    .sort_values("month")
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(context["month"], context["units_sold"], color="lightgray", linewidth=2,
        label=f"Other stores — {top_product} (average)")
ax.plot(focus["month"], focus["units_sold"], "o-", color="#4C72B0", linewidth=2,
        label=f"{top_store} — {top_product} (focus)")
ax.set_xlabel("Month")
ax.set_ylabel("Units Sold")
ax.set_title(f"Focus + Context: {top_store} / {top_product}")
ax.legend()
plt.tight_layout()
plt.show()



**Reflection:** Which interaction-cost tier (eye fixation / hover / click-to-open panel / window
replacement) did you use to first *spot* the anomaly, and which did you use to *investigate* it? Was
using two different tiers the right call, per the lecture's design rule?

_Your answer:_

I first spotted the anomaly using hover on the overview trend chart, which is the cheap hover tier because it lets me identify a suspicious store-product line without opening another view. I then investigated it with a focused chart that keeps the average trend for the same product as context. Using two tiers is appropriate because the lecture's design rule is to use the cheapest interaction that answers each question rather than making every investigation require a more expensive interaction.



## Task 5 — Table Lens: A Scannable Product Performance Table

**Real-world usage:** A manager wants a single table listing every product's total revenue, total
profit, and average rating across all stores — but a plain spreadsheet of 10 rows x many numeric
columns is still hard to compare at a glance across all three metrics simultaneously.

**Why this technique:** Table Lens (Slide 27) maps numeric columns to in-cell bars, so magnitude
differences are visible pre-attentively while the table stays sortable/readable as text.

### Your task
1. Build a summary DataFrame: one row per `product`, columns for total `revenue`, total `profit`, and
   mean `avg_rating`.
2. Sort it by total revenue, descending.
3. Style it as a Table Lens using `.style.bar(...)` — a different bar color per numeric column, as in
   the Demo notebook — and add a meaningful `.set_caption(...)`.
4. Below the table, add **one more cell**: sort the *same* summary table by `profit` instead of
   `revenue`, and re-render the styled table. Do the top rows change order? Note what that tells a
   manager about revenue vs. profitability.


In [ ]:
# TODO: build product_summary with columns: product, revenue (sum), profit (sum), avg_rating (mean)
product_summary = (
    sales_df.groupby("product", as_index=False)
    .agg(revenue=("revenue", "sum"),
         profit=("profit", "sum"),
         avg_rating=("avg_rating", "mean"))
)

# TODO: sort descending by revenue
product_summary = product_summary.sort_values("revenue", ascending=False)

# TODO: style with .style.bar(...) per numeric column + .set_caption(...)
product_summary.style     .bar(subset=["revenue"], color="#4C72B0")     .bar(subset=["profit"], color="#55A868")     .bar(subset=["avg_rating"], color="#C44E52")     .set_caption("Product Performance — Sorted by Total Revenue")


In [ ]:
# TODO: re-sort product_summary by profit descending and re-render the styled table
product_summary_profit = product_summary.sort_values("profit", ascending=False)

product_summary_profit.style     .bar(subset=["revenue"], color="#4C72B0")     .bar(subset=["profit"], color="#55A868")     .bar(subset=["avg_rating"], color="#C44E52")     .set_caption("Product Performance — Sorted by Total Profit")



**Your observation (revenue-sorted vs profit-sorted top rows):**

The top rows can change when the table is sorted by profit rather than revenue. That tells the manager that high sales volume and high profitability are different business outcomes, so a revenue ranking should not be treated as a profitability ranking.



## Task 6 — Assemble a Single-Screen Monitoring Dashboard (open-ended)

**Real-world usage:** All three complaints at once. UrbanCart wants a **single screen** a regional
manager can glance at every morning — no interaction required to read the headline state, per the
Dashboard Patterns slide (Slide 28): critical indicators visible at a glance, with a peripheral-tier
alert that uses color (not subtlety) to flag something that needs attention.

### Your task
Build a 2x2 dashboard (`plt.subplots(2, 2, ...)`) with:

1. **Top-left — KPI summary panel** (`ax.axis("off")` + `ax.text(...)`): total revenue, total profit,
   overall average rating, and number of active stores, computed from `sales_df`.
2. **Top-right — mid-term analysis panel**: revenue by `region` (bar chart).
3. **Bottom-left — long-term trend panel**: total revenue by `month`, across all stores (line chart).
4. **Bottom-right — peripheral alert panel**: check whether **any** `(store_id, product)` pair has a
   month where `units_sold` fell below **25% of its own median** for that pair (this is the anomaly
   you found in Task 4, but write the check generically so it would catch *any* such case, not just
   the one you already know about). Show a red "ALERT — N issue(s) found" indicator if any exist, or
   a green "OK" indicator if not.

This task is intentionally the least scaffolded — you decide the exact `groupby` calls, panel layout
details, and styling. Reuse patterns from Tasks 1-5 and the Demo notebook where they fit, but write
the logic yourself.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Panel 1: KPI summary (top-left) ---
# TODO
total_revenue = sales_df["revenue"].sum()
total_profit = sales_df["profit"].sum()
overall_rating = sales_df["avg_rating"].mean()
active_stores = sales_df["store_id"].nunique()

axes[0, 0].axis("off")
axes[0, 0].text(0.5, 0.72, f"${total_revenue:,.0f}", ha="center", fontsize=24, fontweight="bold")
axes[0, 0].text(0.5, 0.56, "Total Revenue", ha="center", fontsize=11)
axes[0, 0].text(0.5, 0.40, f"${total_profit:,.0f}", ha="center", fontsize=20, fontweight="bold")
axes[0, 0].text(0.5, 0.27, "Total Profit", ha="center", fontsize=10)
axes[0, 0].text(0.5, 0.13, f"Avg Rating: {overall_rating:.2f}   |   Active Stores: {active_stores}",
                 ha="center", fontsize=10)
axes[0, 0].set_title("Key Performance Indicators")

# --- Panel 2: revenue by region (top-right) ---
# TODO
region_revenue = sales_df.groupby("region")["revenue"].sum().sort_values()
axes[0, 1].barh(region_revenue.index, region_revenue.values)
axes[0, 1].set_title("Revenue by Region")
axes[0, 1].set_xlabel("Revenue ($)")

# --- Panel 3: total revenue by month, all stores (bottom-left) ---
# TODO
monthly_revenue = sales_df.groupby("month")["revenue"].sum().sort_index()
axes[1, 0].plot(monthly_revenue.index, monthly_revenue.values, "o-")
axes[1, 0].set_title("Monthly Revenue — All Stores")
axes[1, 0].set_xlabel("Month")
axes[1, 0].set_ylabel("Revenue ($)")
axes[1, 0].tick_params(axis="x", rotation=45)

# --- Panel 4: peripheral alert panel (bottom-right) ---
# TODO: generically detect any (store_id, product) pair with a month below 25% of its own median
#       units_sold, then render a red "ALERT -- N issue(s) found" or green "OK" indicator
pair_medians = sales_df.groupby(["store_id", "product"])["units_sold"].transform("median")
issue_mask = sales_df["units_sold"] < 0.25 * pair_medians
issue_count = int(issue_mask.sum())

axes[1, 1].axis("off")
if issue_count:
    axes[1, 1].text(0.5, 0.5, f"ALERT — {issue_count} issue(s) found",
                    ha="center", va="center", fontsize=20, fontweight="bold", color="#D9534F")
else:
    axes[1, 1].text(0.5, 0.5, "OK",
                    ha="center", va="center", fontsize=24, fontweight="bold", color="#55A868")
axes[1, 1].set_title("Anomaly Monitor")

fig.suptitle("UrbanCart Regional Manager Dashboard", fontsize=14)
plt.tight_layout()
plt.show()



## Reflection

Answer briefly, referring back to the three manager complaints from the scenario intro:

1. Which single technique from this activity do you think gives UrbanCart's managers the *biggest*
   time saving, and why?
2. Which technique was hardest to implement well, and what design trade-off (from the lecture) did you
   have to think about while building it?
3. If you had to add **one more** interaction technique from the lecture that this activity didn't
   cover (e.g. geometric fisheye, magic lens, network zooming, model-based planning), where in this
   retail scenario would it actually be useful, and why?

The coordinated views are likely to save the most time because one selection connects the store overview, category breakdown, and monthly trend without repeated page switching. The hardest part is the coordinated-view and focus+context logic because it requires keeping the overview visible while making the selected subset visually dominant. A useful additional technique would be a magic lens over a dense product-by-store chart, allowing managers to reveal detailed metrics only where they are investigating while preserving the surrounding overview.
